# JAE demo

Denoising a nonlinear neural manifold with the channel-split model (JAE1) and the JEPA model (JAE2).

In [ ]:
import torch

from jae import JAE, simulate_neural_data, pca_denoise, factor_analysis_denoise
from jae.data import train_val_test_split
from jae.metrics import per_channel_vaf, effective_rank, invariance_ratio

clean, noisy, info = simulate_neural_data(
    n_samples=400, n_channels=64, n_timepoints=96,
    latent_dim=6, snr_db=10.0, nonlinear=True, alpha=3.0, seed=0,
)
split = train_val_test_split(noisy, clean=clean, fracs=(0.7, 0.0, 0.3), seed=0)
train, test, test_clean = split['train']['noisy'], split['test']['noisy'], split['test']['clean']
print('shapes:', tuple(noisy.shape))

## JAE1 vs linear baselines

On nonlinear data the channel-split autoencoder should beat PCA and Factor Analysis (on linear data it should not, since PCA is then near-optimal).

In [ ]:
model = JAE(latent_dim=6, backend='jae1', verbose=False)
model.fit(train, epochs=200, batch_size=16)
jae1_vaf = per_channel_vaf(test_clean, model.denoise(test))['mean']
pca_vaf = per_channel_vaf(test_clean, pca_denoise(train, test, k=6))['mean']
fa_vaf = per_channel_vaf(test_clean, factor_analysis_denoise(train, test, k=6))['mean']
print(f'JAE1 VAF={jae1_vaf:.3f}  PCA VAF={pca_vaf:.3f}  FA VAF={fa_vaf:.3f}')

## JAE2 (JEPA): denoising readout and latent-manifold health

The metric panel confirms the learned representation is not collapsed: effective rank well above 1, and a high invariance ratio with a non-zero denominator (a collapsed representation would drive the denominator to zero).

In [ ]:
jepa = JAE(latent_dim=32, backend='jepa', verbose=False, patch_len=8, d_model=64)
jepa.fit(train, epochs=120, batch_size=32)
print('JEPA denoise VAF:', round(per_channel_vaf(test_clean, jepa.denoise(test))['mean'], 3))

jepa.model.eval()
with torch.no_grad():
    out = jepa.model(jepa._apply_standardize(test))
zc, zt = out.z_context.numpy(), out.z_target.numpy()
ir = invariance_ratio(zc, zt)
print(f"effective_rank={effective_rank(zc):.2f}/{zc.shape[1]}  invariance_ratio={ir['ir']:.3f} (denom={ir['denom']:.4f})")